## What are Outliers?
-  data points that differ significantly from other observations in a dataset.
- They are values that lie far outside the typical range of values 
- don't follow the normal pattern of the data.

In [27]:
import pandas as pd
df=pd.read_csv("outliers_dataset.csv")
df

,id,name,age,salary,orders,rating
0,1,Alice,28,45000,15,4.2
1,2,Bob,32,52000,22,7.8
2,3,Charlie,26,48000,18,4.5
3,4,Diana,30,51000,20,4.0
4,5,Eve,25,75000,12,4.3
5,6,Frank,29,49000,16,4.1
6,7,Grace,27,47000,14,3.9
7,8,Hank,31,50000,1,4.4
8,9,Ivy,24,43000,10,4.6
9,10,John,28,44000,13,4.2


- this data set contains some outliers like
- 1: in row 11 age is 150 which is far away from other values
- 2: in row 5 salary 75000 which is most higher than double of some others
- 3: row 8 only 1 order while average is all other are 13 above
- 4: row 2 rating is 7.8 while all other are below 5

# Handling outliers 
- ### there are two main methods for handlin ouliers
- quantile method IQR (inter quantile range) method
- z-score

## 1-IQR method
- in this method we find lower and upper quantile and then delete values that are outside this range

In [28]:
#since i have outliers in each column i need a function to repeat for all
def detect_outliers_single(df, column):
    """Detect outliers in a single column using IQR"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    
    print(f"\n{'='*40}")
    print(f"Column: {column}")
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")
    print(f"Outliers found: {len(outliers)}")
    
    if len(outliers) > 0:
        print("Outlier values:", outliers[column].tolist())
        print("Outlier rows:")
        print(outliers[['id', 'name', column]])
    
    return outliers

In [29]:
# finding outliers for each column
outliers_age = detect_outliers_single(df, 'age')


Column: age
Q1: 26.75
Q3: 30.00
IQR: 3.25
Lower bound: 21.88
Upper bound: 34.88
Outliers found: 1
Outlier values: [150]
Outlier rows:
    id  name  age
10  11  Kate  150


In [30]:
outliers_salary = detect_outliers_single(df, 'salary')


Column: salary
Q1: 45000.00
Q3: 50000.00
IQR: 5000.00
Lower bound: 37500.00
Upper bound: 57500.00
Outliers found: 1
Outlier values: [75000]
Outlier rows:
   id name  salary
4   5  Eve   75000


In [31]:
outliers_orders = detect_outliers_single(df, 'orders')


Column: orders
Q1: 12.75
Q3: 16.25
IQR: 3.50
Lower bound: 7.50
Upper bound: 21.50
Outliers found: 2
Outlier values: [22, 1]
Outlier rows:
   id  name  orders
1   2   Bob      22
7   8  Hank       1


## Removing outliers

In [32]:
def remove_all_outliers(df, columns=None):
    """
    Remove rows that have outliers in any of the specified columns
    If no columns specified, checks all numeric columns
    """
    if columns is None:
        columns = df.select_dtypes(include=['int64', 'float64']).columns
    
    outlier_indices = []
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        col_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_indices.extend(col_outliers.index.tolist())
    
    outlier_indices = list(set(outlier_indices))
    df_cleaned = df.drop(index=outlier_indices)
    
    print(f"Columns checked: {list(columns)}")
    print(f"Outliers removed: {len(outlier_indices)} rows")
    print(f"Remaining rows: {len(df_cleaned)}")
    
    return df_cleaned, outlier_indices

# Use the function
df_cleaned, removed_indices = remove_all_outliers(df, ['age', 'salary', 'orders'])
print("\nCleaned DataFrame:")
print(df_cleaned)

Columns checked: ['age', 'salary', 'orders']
Outliers removed: 4 rows
Remaining rows: 16

Cleaned DataFrame:
    id     name  age  salary  orders  rating
0    1    Alice   28   45000      15     4.2
2    3  Charlie   26   48000      18     4.5
3    4    Diana   30   51000      20     4.0
5    6    Frank   29   49000      16     4.1
6    7    Grace   27   47000      14     3.9
8    9      Ivy   24   43000      10     4.6
9   10     John   28   44000      13     4.2
11  12      Leo   27   49000      17     3.7
12  13      Mia   26   46000      11     4.3
13  14     Nick   29   48000      15     4.0
14  15   Olivia   31   51000      18     4.1
15  16    Peter   28   47000      14     3.8
16  17    Quinn   30   50000      16     4.2
17  18   Rachel   26   44000      12     4.4
18  19      Sam   27   45000      13     4.0
19  20     Tina   29   49000      15     4.3


## 2: Z-Score Method

### Function for detecting

In [33]:
from scipy import stats
import numpy as np

def detect_outliers_zscore(df, column, threshold=3):
    """
    Detect outliers using Z-Score method
    Values with |Z-Score| > threshold are considered outliers
    """
    # Calculate Z-Scores
    z_scores = np.abs(stats.zscore(df[column]))
    
    # Find outliers
    outliers = df[z_scores > threshold]
    
    # Calculate statistics
    mean_val = df[column].mean()
    std_val = df[column].std()
    
    print(f"\n{'='*40}")
    print(f"Column: {column}")
    print(f"Mean: {mean_val:.2f}")
    print(f"Standard Deviation: {std_val:.2f}")
    print(f"Threshold: ±{threshold}")
    print(f"Outliers found: {len(outliers)}")
    
    if len(outliers) > 0:
        print("Outlier values:", outliers[column].tolist())
        print("Outlier rows:")
        print(outliers[['id', 'name', column]])
        
        # Show Z-Scores for outliers
        print("\nZ-Scores for outliers:")
        for idx in outliers.index:
            z = (df.loc[idx, column] - mean_val) / std_val
            print(f"  {df.loc[idx, 'name']}: Z-Score = {z:.2f}")
    
    return outliers

Detecting for each column

In [34]:
# Age column outliers
outliers_age_z = detect_outliers_zscore(df, 'age', threshold=3)


Column: age
Mean: 34.15
Standard Deviation: 27.35
Threshold: ±3
Outliers found: 1
Outlier values: [150]
Outlier rows:
    id  name  age
10  11  Kate  150

Z-Scores for outliers:
  Kate: Z-Score = 4.24


In [ ]:
#  for salary
outliers_salary_z = detect_outliers_zscore(df, 'salary', threshold=3)


Column: salary
Mean: 48700.00
Standard Deviation: 6860.26
Threshold: ±3
Outliers found: 1
Outlier values: [75000]
Outlier rows:
   id name  salary
4   5  Eve   75000

Z-Scores for outliers:
  Eve: Z-Score = 3.83


In [36]:
# Orders column outliers
outliers_orders_z = detect_outliers_zscore(df, 'orders', threshold=3)


Column: orders
Mean: 14.35
Standard Deviation: 4.32
Threshold: ±3
Outliers found: 1
Outlier values: [1]
Outlier rows:
   id  name  orders
7   8  Hank       1

Z-Scores for outliers:
  Hank: Z-Score = -3.09


In [37]:
# Rating column outliers
outliers_rating_z = detect_outliers_zscore(df, 'rating', threshold=3)


Column: rating
Mean: 4.20
Standard Deviation: 1.10
Threshold: ±3
Outliers found: 1
Outlier values: [7.8]
Outlier rows:
   id name  rating
1   2  Bob     7.8

Z-Scores for outliers:
  Bob: Z-Score = 3.27


### Removing Outliers

In [38]:
# Remove outliers using Z-Score method
def remove_outliers_zscore(df, columns, threshold=3):
    """
    Remove rows with outliers using Z-Score method
    """
    original_len = len(df)
    mask = pd.Series([True] * len(df), index=df.index)
    
    for col in columns:
        z_scores = np.abs(stats.zscore(df[col]))
        mask = mask & (z_scores < threshold)
    
    df_cleaned = df[mask].copy()
    removed_rows = original_len - len(df_cleaned)
    
    print(f"\n{'='*40}")
    print("REMOVING OUTLIERS USING Z-SCORE")
    print(f"Columns checked: {columns}")
    print(f"Original rows: {original_len}")
    print(f"Rows removed: {removed_rows}")
    print(f"Remaining rows: {len(df_cleaned)}")
    
    if removed_rows > 0:
        print("\nRemoved rows:")
        removed_indices = df[~mask].index
        print(df.loc[removed_indices, ['id', 'name', 'age', 'salary', 'orders', 'rating']])
    
    return df_cleaned

# Remove outliers from numeric columns
numeric_cols = ['age', 'salary', 'orders', 'rating']
df_cleaned_zscore = remove_outliers_zscore(df, numeric_cols, threshold=3)

print("\nCleaned DataFrame (Z-Score):")
print(df_cleaned_zscore)


REMOVING OUTLIERS USING Z-SCORE
Columns checked: ['age', 'salary', 'orders', 'rating']
Original rows: 20
Rows removed: 4
Remaining rows: 16

Removed rows:
    id  name  age  salary  orders  rating
1    2   Bob   32   52000      22     7.8
4    5   Eve   25   75000      12     4.3
7    8  Hank   31   50000       1     4.4
10  11  Kate  150   41000      15     1.2

Cleaned DataFrame (Z-Score):
    id     name  age  salary  orders  rating
0    1    Alice   28   45000      15     4.2
2    3  Charlie   26   48000      18     4.5
3    4    Diana   30   51000      20     4.0
5    6    Frank   29   49000      16     4.1
6    7    Grace   27   47000      14     3.9
8    9      Ivy   24   43000      10     4.6
9   10     John   28   44000      13     4.2
11  12      Leo   27   49000      17     3.7
12  13      Mia   26   46000      11     4.3
13  14     Nick   29   48000      15     4.0
14  15   Olivia   31   51000      18     4.1
15  16    Peter   28   47000      14     3.8
16  17    Quinn   3